# 🧍 Proyecto Integrador — Estimación de Pose y Despliegue en HF Spaces

**Materiales desarrollados por Matías Barreto, 2026**  
**Tecnicatura Superior en Ciencias de Datos e IA, IFTS24**  
* **Nomenclatura Oficial:** Procesamiento Digital de Imágenes  
* **Nombre de Trabajo:** Laboratorio de Tecnologías de la Imagen Digital  

---

## El proyecto

Este cuaderno es diferente a los anteriores. No es un tutorial paso a paso: es un **proyecto integrador**.

Vamos a partir de un código base funcional para detectar pose corporal con MediaPipe, y desde ahí vamos a construir y desplegar una aplicación web completa. El producto final va a estar publicado en Hugging Face Spaces y versionado en un repositorio de GitHub.

**Al completar este proyecto vamos a haber:**

1. Explorado MediaPipe Pose — la tercera solución de detección que vemos en esta unidad (además de Face Mesh y Hands).
2. Adaptado código de Jupyter a un script `app.py` listo para producción.
3. Desplegado una aplicación de visión artificial accesible desde cualquier navegador.
4. Publicado el código fuente en un repositorio de GitHub.

> ◈ Este cuaderno usa el **Cheatsheet de HF Spaces** como referencia para el despliegue. Conviene tenerlo abierto en otra pestaña: `Extras/Guias/HuggingFace-Spaces/Cheatsheet_Desarrollo_Space.ipynb`

## Microglosario

| Término | Definición | Analogía |
|---|---|---|
| **Pose estimation** | Detección automática de la posición de las articulaciones del cuerpo en una imagen | Como cuando un entrenador marca con stickers los puntos clave del cuerpo de un atleta para analizar su técnica |
| **Keypoint / Punto clave** | Coordenada que representa una articulación o punto anatómico específico (nariz, hombro, rodilla...) | Como los pines de un maniquí articulado: cada uno representa una unión móvil |
| **Visibilidad** | Valor entre 0 y 1 que indica qué tan seguro está el modelo de que ese punto es visible en la imagen | Como la confianza con la que un médico marca un punto en una radiografía: 1.0 = certeza total, 0.0 = pura suposición |
| **`app.py`** | Script Python que contiene la lógica completa de la aplicación, listo para ejecutarse fuera de Jupyter | Como el plano de una casa: en Jupyter dibujamos bocetos, en `app.py` está el plano final para construir |
| **Space (HF)** | Servidor gratuito de Hugging Face que ejecuta y publica aplicaciones Gradio | Como un hosting web especializado en aplicaciones de IA: subís el código y ellos lo sirven al mundo |

## ✦ MediaPipe Pose: los 33 puntos del cuerpo

MediaPipe Pose detecta **33 puntos clave** distribuidos por todo el cuerpo. A diferencia de Face Mesh (478 puntos en el rostro) o Hands (21 puntos en la mano), Pose cubre el esqueleto completo con menos puntos pero mayor alcance anatómico.

```
                [0] nariz
                   |
         [12] ─────┼───── [11]     ← hombros
          |                 |
         [14]              [13]    ← codos
          |                 |
         [16]              [15]    ← muñecas


         [24] ─────────── [23]     ← caderas
          |                 |
         [26]              [25]    ← rodillas
          |                 |
         [28]              [27]    ← tobillos
```

*Nota: los índices siguen la convención MediaPipe — lado derecho de la persona en índices pares, lado izquierdo en impares.*

Cada punto tiene cuatro valores:

| Atributo | Tipo | Descripción |
|---|---|---|
| `x` | float 0–1 | Posición horizontal normalizada |
| `y` | float 0–1 | Posición vertical normalizada |
| `z` | float | Profundidad relativa (aproximada) |
| `visibility` | float 0–1 | Confianza de que el punto es visible |

**Casos de uso:** análisis de postura, entrenamiento deportivo, ergonomía en el trabajo, coreografías, fisioterapia.

## Paso 1 — Instalación

Las mismas herramientas que ya conocemos. Si ya las instalaron en este entorno, la celda termina rápido.

In [1]:
# Instala las dependencias del notebook en el kernel activo.
%pip install mediapipe opencv-python-headless gradio numpy --quiet

import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
import gradio as gr
import cv2
import numpy as np

print("✓ Entorno listo.")
print(f"  mediapipe  {mp.__version__}")
print(f"  gradio     {gr.__version__}")
print(f"  opencv     {cv2.__version__}")



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


c:\Proyectos\rodriguez-carmen-pdi-1c-2026\.venv_vision_aplicada\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Entorno listo.
  mediapipe  0.10.35
  gradio     6.19.0
  opencv     4.13.0


## Código base — detector de pose

La siguiente celda contiene la función central del proyecto. Algunas líneas están completas; otras tienen un `# TODO` donde ustedes van a tener que escribir.

Lean la función completa antes de ejecutarla. Los `# TODO` son parte de la consigna — no los salteen.

In [2]:
# ── IMPORTS ──────────────────────────────────────────────────────────────────

import os                   # para verificar si el archivo del modelo ya existe en disco
import urllib.request       # para descargarlo si no está

import mediapipe as mp
# mediapipe.tasks es la API moderna (0.10+). Se importa en dos partes:
#   - python: contiene BaseOptions (configuración de bajo nivel: ruta del modelo, delegado)
#   - vision: contiene las clases específicas de visión (PoseLandmarker, FaceLandmarker, etc.)
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

import numpy as np          # las imágenes son arrays NumPy; mp y cv2 operan sobre ellos
import cv2                  # dibuja líneas y círculos sobre la imagen
import gradio as gr         # construye la interfaz web interactiva


# ── DESCARGA DEL MODELO ───────────────────────────────────────────────────────

MODELO_PATH = "pose_landmarker_full.task"
# El archivo .task es el modelo en formato TFLite empaquetado por Google.
# "full" significa la variante de mayor precisión (hay también "lite" y "heavy").
MODELO_URL  = (
    "https://storage.googleapis.com/mediapipe-models/"
    "pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task"
)

# Solo descarga si todavía no está en disco; evita re-descargar 6 MB en cada ejecución.
if not os.path.exists(MODELO_PATH):
    print("Descargando modelo de pose corporal (aprox. 6 MB)...")
    urllib.request.urlretrieve(MODELO_URL, MODELO_PATH)
    print("✓ Modelo descargado.")
else:
    print(f"✓ Modelo ya disponible: {MODELO_PATH}")


# ── MAPA DEL ESQUELETO ────────────────────────────────────────────────────────

# MediaPipe devuelve 33 puntos numerados (0 = nariz, 11-12 = hombros, 23-32 = piernas...).
# Esta lista define qué pares de puntos deben conectarse con una línea para
# formar el esqueleto visible. Sin ella, solo veríamos puntos sueltos sin estructura.
POSE_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,7),(0,4),(4,5),(5,6),(6,8),   # cabeza y orejas
    (9,10),(11,12),                                      # boca y línea de hombros
    (11,13),(13,15),(15,17),(15,19),(15,21),(17,19),     # brazo izquierdo
    (12,14),(14,16),(16,18),(16,20),(16,22),(18,20),     # brazo derecho
    (11,23),(12,24),(23,24),                             # tronco
    (23,25),(25,27),(27,29),(27,31),(29,31),             # pierna izquierda
    (24,26),(26,28),(28,30),(28,32),(30,32),             # pierna derecha
]


# ── INICIALIZACIÓN DEL DETECTOR ───────────────────────────────────────────────

# BaseOptions le dice al modelo dónde está el archivo .task.
# También permite elegir delegado de inferencia (CPU o GPU); por defecto usa CPU.
base_options = mp_python.BaseOptions(model_asset_path=MODELO_PATH)

opciones_pose = mp_vision.PoseLandmarkerOptions(
    base_options=base_options,
    num_poses=1,                          # cuántas personas detectar por imagen
    min_pose_detection_confidence=0.5,    # umbral para aceptar una detección como válida
    # 0.5 equilibra sensibilidad y precisión: detecta bien poses parciales sin
    # generar falsos positivos en fondos complejos.
    # Con 0.3 acepta más casos borrosos; con 0.9 solo poses muy claras.
)

# create_from_options carga el modelo en memoria. Es costoso en tiempo;
# por eso se hace UNA sola vez aquí, fuera de la función, y no en cada llamada.
detector_pose = mp_vision.PoseLandmarker.create_from_options(opciones_pose)

print("✓ Detector de Pose inicializado.")


# ── FUNCIÓN PRINCIPAL ─────────────────────────────────────────────────────────

def detectar_pose(imagen_entrada):
    # imagen_entrada es un array NumPy (H, W, 3) en formato RGB,
    # entregado así por Gradio gracias a type="numpy" en gr.Image.
    alto, ancho = imagen_entrada.shape[:2]  # dimensiones en píxeles

    # La Tasks API no acepta arrays NumPy directamente: necesita un objeto mp.Image.
    # SRGB indica que los canales están en orden R-G-B (mismo orden que Gradio entrega).
    imagen_mp = mp.Image(image_format=mp.ImageFormat.SRGB, data=imagen_entrada)

    # detect() ejecuta la red neuronal y devuelve un objeto con los resultados.
    resultado = detector_pose.detect(imagen_mp)

    # Trabajamos sobre una copia para no modificar el array original.
    imagen_anotada = imagen_entrada.copy()

    # pose_landmarks es una lista vacía [] si no se detectó ninguna persona.
    # (La API antigua devolvía None; en Tasks API siempre es lista, posiblemente vacía.)
    if not resultado.pose_landmarks:
        return imagen_anotada, "No se detectó ninguna figura humana en la imagen."

    # resultado.pose_landmarks[0] = primera persona detectada.
    # Cada elemento es un NormalizedLandmark con .x, .y (rango 0.0–1.0),
    # .z (profundidad relativa) y .visibility (confianza de que el punto es visible).
    lista_landmarks = resultado.pose_landmarks[0]

    # ── Dibujar el esqueleto (líneas entre pares de keypoints) ────────────────
    for idx_a, idx_b in POSE_CONNECTIONS:
        lm_a = lista_landmarks[idx_a]
        lm_b = lista_landmarks[idx_b]
        # Solo dibuja el segmento si ambos extremos son suficientemente visibles.
        # Esto evita líneas fantasma cuando un miembro está fuera del encuadre.
        if lm_a.visibility > 0.3 and lm_b.visibility > 0.3:
            # Las coordenadas están normalizadas [0.0, 1.0]; hay que convertirlas a píxeles.
            xa, ya = int(lm_a.x * ancho), int(lm_a.y * alto)
            xb, yb = int(lm_b.x * ancho), int(lm_b.y * alto)
            cv2.line(imagen_anotada, (xa, ya), (xb, yb), (0, 200, 0), 2)
            # color (0,200,0) = verde en RGB; grosor 2 px.

    # ── Dibujar los 33 puntos sobre el esqueleto ──────────────────────────────
    for punto in lista_landmarks:
        if punto.visibility > 0.3:         # mismo filtro de visibilidad
            px = int(punto.x * ancho)
            py = int(punto.y * alto)
            cv2.circle(imagen_anotada, (px, py), 4, (255, 50, 50), -1)
            # Radio 4 px, color rojo (255,50,50), relleno (-1).
            # Se dibuja después de las líneas para que los puntos queden encima.

    # ── Extraer métricas de puntos específicos ────────────────────────────────
    # Referencia de índices: https://ai.google.dev/edge/mediapipe/solutions/vision/pose_landmarker
    punto_hombro_derecho   = lista_landmarks[12]
    punto_hombro_izquierdo = lista_landmarks[11]
    punto_cadera_derecha   = lista_landmarks[24]
    punto_rodilla_derecha  = lista_landmarks[26]
    punto_tobillo_derecho  = lista_landmarks[28]

    # Distancia horizontal entre hombros en coordenadas normalizadas.
    # 1.0 = ancho total de la imagen. Valor típico de frente: ~0.3–0.4.
    # Valor bajo puede indicar que la persona está de perfil.
    distancia_hombros = abs(punto_hombro_derecho.x - punto_hombro_izquierdo.x)
    distancia_hombros_redondeada = round(distancia_hombros, 3)

    # Inclinación lateral: diferencia de altura (eje Y) entre hombros.
    # Y crece hacia ABAJO en coordenadas normalizadas (0 = techo, 1 = piso).
    #   valor positivo → hombro izquierdo más bajo  (inclinada a la izquierda)
    #   valor negativo → hombro derecho más bajo    (inclinada a la derecha)
    #   valor ≈ 0      → hombros nivelados
    inclinacion_hombros = round(punto_hombro_izquierdo.y - punto_hombro_derecho.y, 3)

    # ── Armar el texto de salida ──────────────────────────────────────────────
    lineas = [
        f"Distancia entre hombros (norm.): {distancia_hombros_redondeada}",
        f"Visibilidad hombro derecho: {round(punto_hombro_derecho.visibility, 2)}",
        f"Cadera derecha y={round(punto_cadera_derecha.y, 3)}",
        f"Rodilla derecha y={round(punto_rodilla_derecha.y, 3)}",
        f"Tobillo derecho y={round(punto_tobillo_derecho.y, 3)}",
        f"Inclinación lateral hombros: {inclinacion_hombros:+.3f}",
        # :+.3f fuerza el signo (+/-) para que sea fácil leer la dirección.
    ]
    texto_info = "\n".join(lineas)

    # Devuelve dos valores porque gr.Interface tiene dos outputs definidos.
    return imagen_anotada, texto_info


# ── INTERFAZ GRADIO ───────────────────────────────────────────────────────────

interfaz_pose = gr.Interface(
    fn=detectar_pose,                          # función que se llama con cada imagen
    inputs=gr.Image(label="Fotografía", type="numpy"),
    # type="numpy" es obligatorio: le dice a Gradio que entregue el array NumPy RGB
    # directamente en lugar de la ruta del archivo o un objeto PIL.
    outputs=[
        gr.Image(label="Pose detectada"),      # recibe imagen_anotada (array NumPy)
        gr.Textbox(label="Información de puntos clave"),  # recibe texto_info (string)
    ],
    title="Detector de Pose — MediaPipe",
    description="Subí una imagen de una persona. El modelo va a detectar los 33 puntos del esqueleto.",
    flagging_mode="never",   # desactiva el botón Flag (recolección de ejemplos incorrectos)
)

# Abre la interfaz en el navegador en http://localhost:7860 (o el siguiente puerto libre).
# share=False (por defecto): solo accesible desde esta máquina, sin túnel público.
interfaz_pose.launch()



✓ Modelo ya disponible: pose_landmarker_full.task
✓ Detector de Pose inicializado.
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## ✎ Consigna 1 — Exploración

Antes de pasar al deploy, tomense unos minutos para entender lo que el detector devuelve.

1. **Probá con distintas fotos.** ¿Qué pasa con una imagen donde la persona está de espaldas? ¿Y si hay varias personas? ¿Y con una foto de cuerpo entero vs. una de cintura para arriba?

2. **Cambiá `min_detection_confidence`.** Ponelo en `0.3` y en `0.9`. ¿En qué tipo de imágenes notás la diferencia? ¿Por qué creés que existe ese parámetro?

3. **Completá los `# TODO` de la función `detectar_pose`.** Elegí dos puntos anatómicos adicionales, calculá una métrica propia y agregala al texto de salida. No hay una respuesta correcta única — lo importante es que puedas justificar por qué esa métrica es útil.

In [3]:
import os, urllib.request
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
import numpy as np
import cv2
import gradio as gr

MODELO_PATH = "pose_landmarker_full.task"
MODELO_URL  = (
    "https://storage.googleapis.com/mediapipe-models/"
    "pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task"
)
if not os.path.exists(MODELO_PATH):
    urllib.request.urlretrieve(MODELO_URL, MODELO_PATH)

POSE_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,7),(0,4),(4,5),(5,6),(6,8),
    (9,10),(11,12),
    (11,13),(13,15),(15,17),(15,19),(15,21),(17,19),
    (12,14),(14,16),(16,18),(16,20),(16,22),(18,20),
    (11,23),(12,24),(23,24),
    (23,25),(25,27),(27,29),(27,31),(29,31),
    (24,26),(26,28),(28,30),(28,32),(30,32),
]

def detectar_pose(imagen_entrada, confidence):
    if imagen_entrada is None:
        return None, ""

    alto, ancho = imagen_entrada.shape[:2]

    # El detector se crea aquí adentro porque min_pose_detection_confidence
    # es un parámetro de construcción, no de ejecución. Recrearlo por llamada
    # es aceptable en exploración; en producción se evitaría.
    detector = mp_vision.PoseLandmarker.create_from_options(
        mp_vision.PoseLandmarkerOptions(
            base_options=mp_python.BaseOptions(model_asset_path=MODELO_PATH),
            num_poses=1,
            min_pose_detection_confidence=confidence,
        )
    )

    imagen_mp      = mp.Image(image_format=mp.ImageFormat.SRGB, data=imagen_entrada)
    resultado      = detector.detect(imagen_mp)
    imagen_anotada = imagen_entrada.copy()

    if not resultado.pose_landmarks:
        return imagen_anotada, f"[confidence={confidence:.2f}]  No se detectó ninguna figura humana."

    lista_landmarks = resultado.pose_landmarks[0]

    for idx_a, idx_b in POSE_CONNECTIONS:
        lm_a, lm_b = lista_landmarks[idx_a], lista_landmarks[idx_b]
        if lm_a.visibility > 0.3 and lm_b.visibility > 0.3:
            xa, ya = int(lm_a.x * ancho), int(lm_a.y * alto)
            xb, yb = int(lm_b.x * ancho), int(lm_b.y * alto)
            cv2.line(imagen_anotada, (xa, ya), (xb, yb), (0, 200, 0), 2)

    for punto in lista_landmarks:
        if punto.visibility > 0.3:
            cv2.circle(imagen_anotada,
                       (int(punto.x * ancho), int(punto.y * alto)), 4, (255, 50, 50), -1)

    # ── Puntos originales ─────────────────────────────────────────────────────
    p_hd = lista_landmarks[12]   # hombro derecho
    p_hi = lista_landmarks[11]   # hombro izquierdo
    p_cd = lista_landmarks[24]   # cadera derecha
    p_rd = lista_landmarks[26]   # rodilla derecha
    p_td = lista_landmarks[28]   # tobillo derecho

    distancia_hombros   = round(abs(p_hd.x - p_hi.x), 3)
    inclinacion_hombros = round(p_hi.y - p_hd.y, 3)

    # ── CONSIGNA: dos puntos anatómicos adicionales ───────────────────────────
    # Índice 15 = muñeca izquierda, índice 16 = muñeca derecha.
    # Las muñecas son los extremos distales de los brazos y reflejan directamente
    # la posición de las manos, que varía mucho según la postura.
    p_mi = lista_landmarks[15]   # muñeca izquierda
    p_md = lista_landmarks[16]   # muñeca derecha

    # MÉTRICA: apertura de brazos (distancia horizontal entre muñecas, normalizada).
    # Justificación: con los brazos al costado el valor ronda la distancia entre hombros
    # (~0.3); con los brazos en cruz se acerca a 1.0. Esta métrica permite detectar
    # gestos de apertura sin necesidad de reconstrucción 3D.
    # Se aplica el mismo umbral de visibilidad (0.3) que en el resto del código:
    # si alguna muñeca no es visible, informa en lugar de calcular un valor sin sentido.
    if p_mi.visibility > 0.3 and p_md.visibility > 0.3:
        apertura_brazos = round(abs(p_md.x - p_mi.x), 3)
        linea_apertura  = f"Apertura de brazos (norm.): {apertura_brazos}"
        # Referencia rápida:
        #   < 0.2  → brazos muy juntos / cruzados
        #   0.3–0.5 → posición neutral
        #   > 0.6  → brazos extendidos lateralmente
    else:
        linea_apertura = "Apertura de brazos: muñecas no visibles"
    # ─────────────────────────────────────────────────────────────────────────

    texto_info = "\n".join([
        f"confidence aplicada: {confidence:.2f}",
        f"Distancia entre hombros (norm.): {distancia_hombros}",
        f"Visibilidad hombro derecho: {round(p_hd.visibility, 2)}",
        f"Cadera derecha y={round(p_cd.y, 3)}",
        f"Rodilla derecha y={round(p_rd.y, 3)}",
        f"Tobillo derecho y={round(p_td.y, 3)}",
        f"Inclinación lateral hombros: {inclinacion_hombros:+.3f}",
        linea_apertura,   # ← métrica nueva agregada al texto de salida
    ])

    return imagen_anotada, texto_info


# gr.Blocks porque necesitamos un slider como input adicional,
# algo que gr.Interface no permite de forma directa.
with gr.Blocks(title="Exploración de confidence") as interfaz_explorar:

    gr.Markdown("## Detector de Pose — Exploración de `min_pose_detection_confidence`")
    gr.Markdown("Subí una imagen, mové el slider y hacé clic en **Analizar** para comparar.")

    with gr.Row():
        img_entrada = gr.Image(label="Fotografía", type="numpy")
        img_salida  = gr.Image(label="Pose detectada")

    slider_confidence = gr.Slider(
        minimum=0.1, maximum=0.99, step=0.05, value=0.5,
        label="min_pose_detection_confidence",
        info="0.1 = acepta casi todo  |  0.99 = solo poses muy claras"
    )

    salida_texto   = gr.Textbox(label="Métricas")
    boton_analizar = gr.Button("Analizar", variant="primary")

    boton_analizar.click(
        fn=detectar_pose,
        inputs=[img_entrada, slider_confidence],
        outputs=[img_salida, salida_texto],
    )

interfaz_explorar.launch()


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [4]:
import os, urllib.request
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
import numpy as np
import cv2
import gradio as gr

MODELO_PATH = "pose_landmarker_full.task"
MODELO_URL  = (
    "https://storage.googleapis.com/mediapipe-models/"
    "pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task"
)
if not os.path.exists(MODELO_PATH):
    urllib.request.urlretrieve(MODELO_URL, MODELO_PATH)

POSE_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,7),(0,4),(4,5),(5,6),(6,8),
    (9,10),(11,12),
    (11,13),(13,15),(15,17),(15,19),(15,21),(17,19),
    (12,14),(14,16),(16,18),(16,20),(16,22),(18,20),
    (11,23),(12,24),(23,24),
    (23,25),(25,27),(27,29),(27,31),(29,31),
    (24,26),(26,28),(28,30),(28,32),(30,32),
]

def detectar_pose(imagen_entrada, confidence):
    if imagen_entrada is None:
        return None, ""

    alto, ancho = imagen_entrada.shape[:2]

    # El detector se crea aquí adentro porque min_pose_detection_confidence
    # es un parámetro de construcción, no de ejecución. Recrearlo por llamada
    # es aceptable en exploración; en producción se evitaría.
    detector = mp_vision.PoseLandmarker.create_from_options(
        mp_vision.PoseLandmarkerOptions(
            base_options=mp_python.BaseOptions(model_asset_path=MODELO_PATH),
            num_poses=1,
            min_pose_detection_confidence=confidence,
        )
    )

    imagen_mp     = mp.Image(image_format=mp.ImageFormat.SRGB, data=imagen_entrada)
    resultado     = detector.detect(imagen_mp)
    imagen_anotada = imagen_entrada.copy()

    if not resultado.pose_landmarks:
        return imagen_anotada, f"[confidence={confidence:.2f}]  No se detectó ninguna figura humana."

    lista_landmarks = resultado.pose_landmarks[0]

    for idx_a, idx_b in POSE_CONNECTIONS:
        lm_a, lm_b = lista_landmarks[idx_a], lista_landmarks[idx_b]
        if lm_a.visibility > 0.3 and lm_b.visibility > 0.3:
            xa, ya = int(lm_a.x * ancho), int(lm_a.y * alto)
            xb, yb = int(lm_b.x * ancho), int(lm_b.y * alto)
            cv2.line(imagen_anotada, (xa, ya), (xb, yb), (0, 200, 0), 2)

    for punto in lista_landmarks:
        if punto.visibility > 0.3:
            cv2.circle(imagen_anotada,
                       (int(punto.x * ancho), int(punto.y * alto)), 4, (255, 50, 50), -1)

    p_hd = lista_landmarks[12]
    p_hi = lista_landmarks[11]
    p_cd = lista_landmarks[24]
    p_rd = lista_landmarks[26]
    p_td = lista_landmarks[28]

    distancia_hombros   = round(abs(p_hd.x - p_hi.x), 3)
    inclinacion_hombros = round(p_hi.y - p_hd.y, 3)

    texto_info = "\n".join([
        f"confidence aplicada: {confidence:.2f}",
        f"Distancia entre hombros (norm.): {distancia_hombros}",
        f"Visibilidad hombro derecho: {round(p_hd.visibility, 2)}",
        f"Cadera derecha y={round(p_cd.y, 3)}",
        f"Rodilla derecha y={round(p_rd.y, 3)}",
        f"Tobillo derecho y={round(p_td.y, 3)}",
        f"Inclinación lateral hombros: {inclinacion_hombros:+.3f}",
    ])

    return imagen_anotada, texto_info


# gr.Blocks porque necesitamos un slider como input adicional,
# algo que gr.Interface no permite de forma directa.
with gr.Blocks(title="Exploración de confidence") as interfaz_explorar:

    gr.Markdown("## Detector de Pose — Exploración de `min_pose_detection_confidence`")
    gr.Markdown("Subí una imagen, mové el slider y hacé clic en **Analizar** para comparar.")

    with gr.Row():
        img_entrada = gr.Image(label="Fotografía", type="numpy")
        img_salida  = gr.Image(label="Pose detectada")

    slider_confidence = gr.Slider(
        minimum=0.1, maximum=0.99, step=0.05, value=0.5,
        label="min_pose_detection_confidence",
        info="0.1 = acepta casi todo  |  0.99 = solo poses muy claras"
    )

    salida_texto  = gr.Textbox(label="Métricas")
    boton_analizar = gr.Button("Analizar", variant="primary")

    boton_analizar.click(
        fn=detectar_pose,
        inputs=[img_entrada, slider_confidence],
        outputs=[img_salida, salida_texto],
    )

interfaz_explorar.launch()


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


| Punto anatómico | Índice MediaPipe | Justificación |
|---|---|---|
| Muñeca izquierda | 15 | Extremo distal del brazo izquierdo — sensible a cualquier gesto o postura de brazos |
| Muñeca derecha | 16 | Extremo distal del brazo derecho |
| **Apertura de brazos** (métrica) | — | Distancia horizontal normalizada entre muñecas. Cuantifica si la persona tiene los brazos abiertos, al costado o cruzados sin necesitar geometría 3D. Rango orientativo: < 0.2 brazos juntos · 0.3–0.5 posición neutral · > 0.6 brazos extendidos |


## De Jupyter a `app.py`

Un cuaderno Jupyter es un excelente entorno de exploración, pero no es lo que esperan los servidores de producción. Para desplegar en Hugging Face Spaces necesitamos un script Python clásico: `app.py`.

La lógica es la misma que ya conocemos de la unidad anterior — **arquitectura de 3 capas**:

```
┌──────────────────────────────────────────┐
│  CAPA 1 — Data Layer                     │
│  Carga única del modelo en memoria       │
│  (se ejecuta una sola vez al iniciar)    │
└──────────────────┬───────────────────────┘
                   ↓
┌──────────────────────────────────────────┐
│  CAPA 2 — Business Logic                 │
│  Función que procesa cada imagen         │
│  (se llama cada vez que llega una foto)  │
└──────────────────┬───────────────────────┘
                   ↓
┌──────────────────────────────────────────┐
│  CAPA 3 — Presentation Layer             │
│  Interfaz Gradio declarada con Blocks    │
│  (define cómo se ve la app)              │
└──────────────────────────────────────────┘
```

> ◈ Para los detalles de git y deploy, usen el Cheatsheet:
> `Extras/Guias/HuggingFace-Spaces/Cheatsheet_Desarrollo_Space.ipynb`

La celda siguiente genera los archivos de la aplicación directamente desde el cuaderno.

In [5]:
# Esta celda genera los archivos del proyecto en una carpeta local.
# Una vez generados, esa carpeta se sube a Hugging Face Spaces con git.

import os

# Cambia este nombre por el que quieras darle a tu Space en Hugging Face.
NOMBRE_PROYECTO = "mi-pose-app"

os.makedirs(NOMBRE_PROYECTO, exist_ok=True)
print(f"\u2713 Carpeta creada: {NOMBRE_PROYECTO}/")

APP_PY_CONTENT = '# app.py -- Detector de Pose con MediaPipe (Tasks API)\n# Estructura: 3 capas (Data Layer / Business Logic / Presentation Layer)\n\nimport mediapipe as mp\nfrom mediapipe.tasks import python as mp_python\nfrom mediapipe.tasks.python import vision as mp_vision\nimport gradio as gr\nimport numpy as np\nimport cv2\nimport os\nimport urllib.request\n\n\n# -------------------------------------------------------------------------\n# CAPA 1 -- DATA LAYER\n# El modelo se descarga y carga una sola vez cuando arranca la aplicacion.\n# Si lo cargaramos dentro de la funcion, cada request esperaria la carga.\n# -------------------------------------------------------------------------\n\nMODELO_PATH = "pose_landmarker_full.task"\nMODELO_URL  = (\n    "https://storage.googleapis.com/mediapipe-models/"\n    "pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task"\n)\n\nif not os.path.exists(MODELO_PATH):\n    urllib.request.urlretrieve(MODELO_URL, MODELO_PATH)\n\nPOSE_CONNECTIONS = [\n    (0,1),(1,2),(2,3),(3,7),(0,4),(4,5),(5,6),(6,8),\n    (9,10),(11,12),\n    (11,13),(13,15),(15,17),(15,19),(15,21),(17,19),\n    (12,14),(14,16),(16,18),(16,20),(16,22),(18,20),\n    (11,23),(12,24),(23,24),\n    (23,25),(25,27),(27,29),(27,31),(29,31),\n    (24,26),(26,28),(28,30),(28,32),(30,32),\n]\n\n# 0.5 equilibra sensibilidad y precision para uso general.\ndetector_pose = mp_vision.PoseLandmarker.create_from_options(\n    mp_vision.PoseLandmarkerOptions(\n        base_options=mp_python.BaseOptions(model_asset_path=MODELO_PATH),\n        num_poses=1,\n        min_pose_detection_confidence=0.5,\n    )\n)\n\n\n# -------------------------------------------------------------------------\n# CAPA 2 -- BUSINESS LOGIC\n# Toda la logica de procesamiento vive aca, desacoplada de la interfaz.\n# -------------------------------------------------------------------------\n\ndef detectar_pose(imagen_entrada):\n    alto, ancho = imagen_entrada.shape[:2]\n\n    imagen_mp  = mp.Image(image_format=mp.ImageFormat.SRGB, data=imagen_entrada)\n    resultado  = detector_pose.detect(imagen_mp)\n    imagen_anotada = imagen_entrada.copy()\n\n    if not resultado.pose_landmarks:\n        return imagen_anotada, "No se detecto ninguna figura humana en la imagen."\n\n    lista_landmarks = resultado.pose_landmarks[0]\n\n    for idx_a, idx_b in POSE_CONNECTIONS:\n        lm_a = lista_landmarks[idx_a]\n        lm_b = lista_landmarks[idx_b]\n        if lm_a.visibility > 0.3 and lm_b.visibility > 0.3:\n            xa, ya = int(lm_a.x * ancho), int(lm_a.y * alto)\n            xb, yb = int(lm_b.x * ancho), int(lm_b.y * alto)\n            cv2.line(imagen_anotada, (xa, ya), (xb, yb), (0, 200, 0), 2)\n\n    for punto in lista_landmarks:\n        if punto.visibility > 0.3:\n            cv2.circle(imagen_anotada, (int(punto.x * ancho), int(punto.y * alto)), 4, (255, 50, 50), -1)\n\n    punto_hombro_derecho   = lista_landmarks[12]\n    punto_hombro_izquierdo = lista_landmarks[11]\n    punto_cadera_derecha   = lista_landmarks[24]\n    punto_rodilla_derecha  = lista_landmarks[26]\n    punto_tobillo_derecho  = lista_landmarks[28]\n\n    distancia_hombros   = round(abs(punto_hombro_derecho.x - punto_hombro_izquierdo.x), 3)\n    inclinacion_hombros = round(punto_hombro_izquierdo.y - punto_hombro_derecho.y, 3)\n\n    lineas = [\n        f"Distancia entre hombros (norm.): {distancia_hombros}",\n        f"Visibilidad hombro derecho: {round(punto_hombro_derecho.visibility, 2)}",\n        f"Cadera derecha y={round(punto_cadera_derecha.y, 3)}",\n        f"Rodilla derecha y={round(punto_rodilla_derecha.y, 3)}",\n        f"Tobillo derecho y={round(punto_tobillo_derecho.y, 3)}",\n        f"Inclinacion lateral hombros: {inclinacion_hombros:+.3f}",\n    ]\n    texto_info = "\\n".join(lineas)\n\n    return imagen_anotada, texto_info\n\n\n# -------------------------------------------------------------------------\n# CAPA 3 -- PRESENTATION LAYER\n# -------------------------------------------------------------------------\n\nwith gr.Blocks(title="Detector de Pose") as aplicacion:\n\n    gr.Markdown("## Detector de Pose corporal -- MediaPipe")\n    gr.Markdown(\n        "Subi una imagen de una persona y el modelo va a detectar "\n        "los 33 puntos clave del esqueleto corporal."\n    )\n\n    with gr.Row():\n        entrada_imagen = gr.Image(label="Fotografia", type="numpy")\n\n    with gr.Row():\n        salida_imagen = gr.Image(label="Pose detectada")\n        salida_texto  = gr.Textbox(label="Informacion de puntos clave")\n\n    boton_analizar = gr.Button("Analizar pose", variant="primary")\n\n    boton_analizar.click(\n        fn=detectar_pose,\n        inputs=entrada_imagen,\n        outputs=[salida_imagen, salida_texto],\n    )\n\n\nif __name__ == "__main__":\n    aplicacion.launch()\n'

ruta_app = os.path.join(NOMBRE_PROYECTO, 'app.py')
with open(ruta_app, 'w', encoding='utf-8') as f:
    f.write(APP_PY_CONTENT)

print("\u2713 app.py generado.")
print("  Cambia NOMBRE_PROYECTO antes de hacer el deploy.")


✓ Carpeta creada: mi-pose-app/
✓ app.py generado.
  Cambia NOMBRE_PROYECTO antes de hacer el deploy.


In [6]:
import os

dependencias = [
    "gradio>=4.0.0",
    "mediapipe>=0.10.13",
    "opencv-python-headless>=4.8.0",
    "numpy>=1.24.0",
]

contenido_requirements = "\n".join(dependencias)

ruta_requirements = os.path.join(NOMBRE_PROYECTO, 'requirements.txt')
with open(ruta_requirements, 'w', encoding='utf-8') as f:
    f.write(contenido_requirements)

print("✓ requirements.txt generado:")
print()
for dep in dependencias:
    print(f"  {dep}")

print()
print("Archivos del proyecto:")
for nombre_archivo in os.listdir(NOMBRE_PROYECTO):
    print(f"  {NOMBRE_PROYECTO}/{nombre_archivo}")


✓ requirements.txt generado:

  gradio>=4.0.0
  mediapipe>=0.10.13
  opencv-python-headless>=4.8.0
  numpy>=1.24.0

Archivos del proyecto:
  mi-pose-app/app.py
  mi-pose-app/requirements.txt


## ✎ Consigna 2 — La interfaz

El `app.py` que generamos tiene varios `# TODO` pendientes. La consigna es completarlos hasta tener una aplicación que corra sin errores.

**Pasos:**

1. Abrí `app.py` en VS Code (o cualquier editor).

2. **Completá la función `detectar_pose`** pegando la versión final que construiste en la Consigna 1 — con las métricas propias incluidas.

3. **Completá los componentes de la interfaz** (`entrada_imagen`, `salida_imagen`, `salida_texto`). Usen el Cheatsheet de Extras como referencia para ver los componentes disponibles.

4. **Probá la app localmente** desde la terminal:
   ```bash
   cd mi-pose-app
   python app.py
   ```
   Si abre el navegador y funciona, están listos para el deploy.

> ◈ **¿La función no devuelve lo que esperan?** Revisá que los componentes en `outputs=` coincidan exactamente con los valores que devuelve `detectar_pose` (imagen + texto, en ese orden).

## ✎ Consigna 3 — El despliegue

Con la app funcionando localmente, es momento de publicarla. El proceso completo está detallado en el Cheatsheet de Extras — acá va el resumen:

### En Hugging Face Spaces

1. Entrá a [huggingface.co/new-space](https://huggingface.co/new-space)
2. Elegí un nombre para el Space (puede coincidir con `NOMBRE_PROYECTO`)
3. Seleccioná **SDK: Gradio** y **Hardware: CPU free**
4. Seguí los comandos de git del Cheatsheet para vincular y subir los archivos:
   ```bash
   git init
   git add .
   git commit -m 'feat: detector de pose con MediaPipe'
   git remote add origin https://huggingface.co/spaces/TU_USUARIO/TU_SPACE
   git branch -M main
   git push -u origin main
   ```

### En GitHub

5. Creá un repositorio nuevo en [github.com/new](https://github.com/new)
6. Vinculá el mismo proyecto con un segundo remote:
   ```bash
   git remote add github https://github.com/TU_USUARIO/TU_REPO
   git push github main
   ```

> ◈ El Space en HF va a quedar público y accesible por URL. Compartí el link cuando esté desplegado.

## ✎ Para pensar

Una vez que la aplicación esté desplegada, respondé estas preguntas:

1. **Sobre el modelo:** MediaPipe Pose fue entrenado con millones de imágenes. Sin embargo, en algunas fotos falla o detecta puntos en posiciones incorrectas. ¿En qué tipo de imágenes notaste más errores? ¿A qué factores creés que se debe?

2. **Sobre la arquitectura:** El `app.py` separa la carga del modelo (Capa 1) de la función de procesamiento (Capa 2). ¿Qué pasaría si cargáramos el modelo *dentro* de `detectar_pose`, en lugar de hacerlo una sola vez al inicio? ¿Por qué eso sería un problema en producción?

3. **Sobre el deploy:** Comparando el flujo que siguieron hoy (Jupyter → `app.py` → HF Spaces + GitHub) con cómo venían trabajando, ¿qué ventajas concretas tiene este proceso? ¿Qué parte les resultó más difícil de entender o ejecutar?